# Demografik Alt Grup ve Adalet Sınıflandırma Analizi
**Chest X-Ray Tiered Classification · Algoritmik Adalet (Fairness) ve Eşitlik**

Bu defter, kademeli Chest X-ray Pneumothorax sistemimizin demografik gruplara göre performans kararlılığını analiz eder. İncelediğimiz alt gruplar:
- **Yaş (Age Bins):** <40, 40-60, 60-80, 80+
- **Cinsiyet (Gender):** Erkek (M) / Kadın (F)
- **Görüntüleme Yönü (View Position):** AP (Anteroposterior) / PA (Posteroanterior)

Her bir alt grup için AUC-ROC ve ECE değerlerini hesaplayıp, DeLong istatistiksel testleri ile adaletli dağılım sergileyip sergilemediğini kontrol ediyoruz.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from core.evaluation.stats import delong_test

# Simüle edilmiş demografik test veri kümesi oluşturma
np.random.seed(42)
n_samples = 600

# Rastgele cinsiyet, yaş ve çekim yönü
genders = np.random.choice(['M', 'F'], size=n_samples, p=[0.52, 0.48])
ages = np.random.randint(18, 92, size=n_samples)
views = np.random.choice(['PA', 'AP'], size=n_samples, p=[0.60, 0.40])

y_true = (np.random.rand(n_samples) < 0.35).astype(int)

# Kademeli sistem tahmin olasılıkları
probs = y_true * 0.65 + (1 - y_true) * 0.18 + np.random.normal(0, 0.16, n_samples)
probs = np.clip(probs, 0.01, 0.99)

df = pd.DataFrame({
    'y_true': y_true,
    'probs': probs,
    'gender': genders,
    'age': ages,
    'view': views
})

# Yaş gruplarını kategorize et
def bin_age(age):
    if age < 40: return '<40'
    elif age <= 60: return '40-60'
    elif age <= 80: return '60-80'
    else: return '80+'

df['age_group'] = df['age'].apply(bin_age)

### Cinsiyete Göre Sınıflandırma Adaleti Analizi

In [ ]:
for gender in ['M', 'F']:
    sub = df[df['gender'] == gender]
    auc = roc_auc_score(sub['y_true'], sub['probs'])
    print(f"Cinsiyet {gender} AUC-ROC: {auc:.4f} (Örnek sayısı: {len(sub)})")

# DeLong testi ile erkek vs kadın AUC-ROC istatistiksel fark analizi
sub_m = df[df['gender'] == 'M']
sub_f = df[df['gender'] == 'F']
# Boyutları dengeleyerek p-value hesapla
min_len = min(len(sub_m), len(sub_f))
p_val = delong_test(
    sub_m['y_true'].values[:min_len], 
    sub_m['probs'].values[:min_len], 
    sub_f['probs'].values[:min_len]
)
print(f"\nCinsiyet grupları arası DeLong Testi p-değeri: {p_val:.4f}")
if p_val > 0.05:
    print("Sonuç: İki grup arasında istatistiksel olarak anlamlı bir performans farkı/taraf tutma (bias) bulunamamıştır.")
else:
    print("Sonuç: Gruplar arasında anlamlı performans farkı tespit edilmiştir.")

### Yaş Gruplarına Göre Performans Karşılaştırması

In [ ]:
age_groups = ['<40', '40-60', '60-80', '80+']
aucs = []
for group in age_groups:
    sub = df[df['age_group'] == group]
    auc = roc_auc_score(sub['y_true'], sub['probs'])
    aucs.append(auc)

plt.figure(figsize=(8, 5))
plt.bar(age_groups, aucs, color='teal', alpha=0.75, edgecolor='black')
plt.ylim(0.70, 1.0)
plt.ylabel('AUC-ROC Skoru')
plt.xlabel('Yaş Bins')
plt.title('Yaş Gruplarına Göre Tanı Başarısı Eşitliği')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()